# R(2+1)D EF Grad-CAM

This notebook generates standard 3D Grad-CAM explanations for the trained R(2+1)D EF regression baseline. It reuses the deterministic cardiac-cycle preprocessing from notebook 13 and writes outputs in a layout similar to the ConvLSTM EF Grad-CAM notebook. Grad-CAM targets the predicted EF scalar, not the loss or ground-truth EF.


In [ ]:
from pathlib import Path
import json
import os
import sys
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# Kaggle-compatible project discovery. Override PROJECT_ROOT if your code dataset has a different path.
PROJECT_ROOT = Path(os.environ.get(
    "PROJECT_ROOT",
    "/kaggle/input/datasets/sooahnoh/echonet-code-updated-3" if Path("/kaggle/input").exists() else ".",
)).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cardiac_cycle_dataset import (
    CardiacCycleWindowConfig,
    EchoNetCardiacCycleDataset,
    build_cardiac_cycle_manifest,
)
from src.gradcam_r2plus1d import (
    make_r2plus1d_heatmap_only_figure,
    make_r2plus1d_motion_trace_overlay,
    make_r2plus1d_overlay_figure,
    make_r2plus1d_overlay_video,
    r2plus1d_cam_metric_table,
    r2plus1d_ef_gradcam,
    run_r2plus1d_normal_inference,
    save_r2plus1d_gradcam_npz,
    select_representative_videos,
)
from src.r2plus1d_ef import R2Plus1DEFRegressor
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", "/kaggle/input/datasets/jiyoonoh24/echonet-dynamic/EchoNet-Dynamic" if Path("/kaggle/input").exists() else PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
VIDEOS_DIR = RAW_DIR / "Videos"
TRAINED_RUN_DIR = Path(os.environ.get("R2PLUS1D_TRAINED_RUN_DIR", "/kaggle/input/datasets/sooahnoh/r2plus1d-ef-baseline-training" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "r2plus1d_ef_baseline_training_07_22"))
BASELINE_PREDICTIONS_CSV = Path(os.environ.get("R2PLUS1D_BASELINE_PREDICTIONS_CSV", str(TRAINED_RUN_DIR / "manifests" / "test_predictions.csv")))

RUN_DIR = Path(os.environ.get("RUN_DIR", "/kaggle/working/outputs/runs/r2plus1d_gradcam" if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "r2plus1d_gradcam"))
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MANIFEST_DIR = RUN_DIR / "manifests"
for directory in [RUN_DIR, CHECKPOINT_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Trained R(2+1)D run/checkpoint directory: {TRAINED_RUN_DIR}")
print(f"Grad-CAM output directory: {RUN_DIR}")


## Configuration


In [ ]:
RUN_MODE = "smoke"  # change to "full" for 10 selected test videos

DEFAULT_CONFIG = {
    "seed": 42,
    "clip_length": 32,
    "image_size": [112, 112],
    "cycle_scale": 2.0,
    "min_window_frames": 32,
    "force_include_ed_es": True,
    "prefer_true_cycle": True,
    "batch_size_for_prediction": 4,
    "num_workers": 2,
    "dropout": 0.2,
    "hidden_dim": 256,
    "pretrained_backbone": False,
    "smoke_max_test_videos": 64,
    "smoke_gradcam_samples": 2,
    "full_gradcam_samples": 10,
    "gradcam_layers": ["layer1", "layer2", "layer4"],
    "expected_native_t_by_layer": {"layer1": 32, "layer2": 16, "layer4": 4},
    "overlay_alpha": 0.45,
    "overlay_alpha_by_layer": {"layer1": 0.55, "layer2": 0.45, "layer4": 0.35},
    "motion_trace_contour_level_by_layer": {"layer1": None, "layer2": 0.5, "layer4": 0.5},
    "video_fps": 6,
}

trained_config_path = TRAINED_RUN_DIR / "config.json"
if trained_config_path.exists():
    trained_config = json.loads(trained_config_path.read_text())
    for key in [
        "seed", "clip_length", "image_size", "cycle_scale", "min_window_frames",
        "force_include_ed_es", "prefer_true_cycle", "dropout", "hidden_dim",
    ]:
        if key in trained_config:
            DEFAULT_CONFIG[key] = trained_config[key]

config = dict(DEFAULT_CONFIG)
config["run_mode"] = RUN_MODE
GRADCAM_SAMPLE_COUNT = int(config["smoke_gradcam_samples"] if RUN_MODE == "smoke" else config["full_gradcam_samples"])
set_seed(int(config["seed"]))
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump({**config, "gradcam_sample_count": GRADCAM_SAMPLE_COUNT}, f, indent=2)
config


## Build Deterministic Cardiac-Cycle Test Set


In [ ]:
assert (RAW_DIR / "FileList.csv").exists(), f"Missing FileList.csv under {RAW_DIR}"
assert (RAW_DIR / "VolumeTracings.csv").exists(), f"Missing VolumeTracings.csv under {RAW_DIR}"
assert VIDEOS_DIR.exists(), f"Missing Videos directory: {VIDEOS_DIR}"
file_list, volume_tracings = load_echonet_tables(RAW_DIR)
window_config = CardiacCycleWindowConfig(
    clip_length=int(config["clip_length"]),
    image_size=tuple(config["image_size"]),
    cycle_scale=float(config["cycle_scale"]),
    min_window_frames=int(config["min_window_frames"]),
    force_include_ed_es=bool(config["force_include_ed_es"]),
    prefer_true_cycle=bool(config["prefer_true_cycle"]),
)
manifest_df = build_cardiac_cycle_manifest(
    file_list=file_list,
    volume_tracings=volume_tracings,
    videos_dir=VIDEOS_DIR,
    config=window_config,
    max_videos=None,
)
assert not manifest_df.empty, "No cardiac-cycle clips were found."
train_manifest = manifest_df[manifest_df["split"] == "TRAIN"].reset_index(drop=True)
test_manifest = manifest_df[manifest_df["split"] == "TEST"].reset_index(drop=True)
assert len(train_manifest) and len(test_manifest), {"train": len(train_manifest), "test": len(test_manifest)}

# Match notebook 13: EF normalization comes from the training split.
ef_mean = float(train_manifest["ef"].mean())
ef_std = float(train_manifest["ef"].std(ddof=0))
assert ef_std > 0, "Training EF standard deviation is zero."

if RUN_MODE == "smoke":
    test_manifest = test_manifest.iloc[: int(config["smoke_max_test_videos"])].reset_index(drop=True)

test_manifest.to_csv(MANIFEST_DIR / "test_manifest.csv", index=False)
test_dataset = EchoNetCardiacCycleDataset(test_manifest, image_size=tuple(config["image_size"]), ef_mean=ef_mean, ef_std=ef_std)
prediction_loader = DataLoader(
    test_dataset,
    batch_size=int(config["batch_size_for_prediction"]),
    shuffle=False,
    num_workers=int(config["num_workers"]),
    pin_memory=torch.cuda.is_available(),
    persistent_workers=int(config["num_workers"]) > 0,
)
gradcam_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)
sample = test_dataset[0]
assert tuple(sample["video"].shape) == (3, int(config["clip_length"]), *tuple(config["image_size"])), tuple(sample["video"].shape)
print(f"Active test videos: {len(test_dataset):,}")
print(f"EF normalization: mean={ef_mean:.3f}, std={ef_std:.3f}")
print({k: tuple(v.shape) for k, v in sample.items() if torch.is_tensor(v) and v.ndim > 0})


## Load R(2+1)D Checkpoint


In [ ]:
def find_checkpoint(root: Path) -> Path:
    env_path = os.environ.get("R2PLUS1D_CHECKPOINT_PATH")
    if env_path:
        path = Path(env_path)
        assert path.exists(), f"R2PLUS1D_CHECKPOINT_PATH does not exist: {path}"
        return path
    candidates = [
        root / "checkpoints" / "best_ef_mae_model.pt",
        root / "best_ef_mae_model.pt",
        root / "checkpoints" / "final_model.pt",
        root / "final_model.pt",
    ]
    for path in candidates:
        if path.exists():
            return path
    recursive = sorted(root.rglob("*.pt")) + sorted(root.rglob("*.pth")) + sorted(root.rglob("*.ckpt"))
    if recursive:
        return recursive[0]
    raise FileNotFoundError(f"No checkpoint found under {root}. Set R2PLUS1D_CHECKPOINT_PATH explicitly.")

checkpoint_path = find_checkpoint(TRAINED_RUN_DIR)
checkpoint = torch.load(checkpoint_path, map_location="cpu")
state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
if any(key.startswith("module.") for key in state_dict):
    state_dict = {key.removeprefix("module."): value for key, value in state_dict.items()}

if isinstance(checkpoint, dict):
    ckpt_config = checkpoint.get("config", {}) or {}
    ef_mean = float(checkpoint.get("ef_mean", ef_mean))
    ef_std = float(checkpoint.get("ef_std", ef_std))
else:
    ckpt_config = {}

model = R2Plus1DEFRegressor(
    pretrained=False,
    dropout=float(ckpt_config.get("dropout", config["dropout"])),
    hidden_dim=int(ckpt_config.get("hidden_dim", config["hidden_dim"])),
).to(device)
load_result = model.load_state_dict(state_dict, strict=True)
model.eval()
checkpoint_metadata = {
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_keys": sorted(list(checkpoint.keys())) if isinstance(checkpoint, dict) else [],
    "epoch": checkpoint.get("epoch") if isinstance(checkpoint, dict) else None,
    "metrics": checkpoint.get("metrics") if isinstance(checkpoint, dict) else None,
    "ef_mean": ef_mean,
    "ef_std": ef_std,
}
with (MANIFEST_DIR / "checkpoint_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, default=str)
print(json.dumps(checkpoint_metadata, indent=2, default=str))


## Normal Test Inference and Baseline Verification


In [ ]:
predictions_df, dataset_metrics = run_r2plus1d_normal_inference(model, prediction_loader, device, ef_mean, ef_std)
predictions_df.to_csv(MANIFEST_DIR / "normal_test_predictions.csv", index=False)
with (RUN_DIR / "test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(dataset_metrics, f, indent=2)
print(json.dumps(dataset_metrics, indent=2))

if BASELINE_PREDICTIONS_CSV.exists():
    baseline_df = pd.read_csv(BASELINE_PREDICTIONS_CSV)
    compare = predictions_df.merge(
        baseline_df[["video_id", "ef_pred"]].rename(columns={"ef_pred": "baseline_ef_pred"}),
        on="video_id",
        how="inner",
    )
    max_abs_diff = float((compare["ef_pred"] - compare["baseline_ef_pred"]).abs().max()) if len(compare) else float("nan")
    print({"baseline_prediction_rows_matched": int(len(compare)), "max_abs_prediction_difference": max_abs_diff})
    assert len(compare) > 0, "No overlapping video IDs with baseline predictions."
    assert max_abs_diff < 1e-3, f"Predictions do not match baseline inference: max diff={max_abs_diff}"
else:
    print(f"Baseline predictions CSV not found at {BASELINE_PREDICTIONS_CSV}; saved fresh normal inference predictions for this run.")

display(predictions_df.head())


## Select Grad-CAM Videos


In [ ]:
selected_video_ids = select_representative_videos(predictions_df, count=GRADCAM_SAMPLE_COUNT)
selected_df = predictions_df[predictions_df["video_id"].isin(selected_video_ids)].copy()
selected_df["selection_rank"] = selected_df["video_id"].map({video_id: rank for rank, video_id in enumerate(selected_video_ids)})
selected_df = selected_df.sort_values("selection_rank").reset_index(drop=True)
selected_df.to_csv(MANIFEST_DIR / "selected_gradcam_videos.csv", index=False)
video_id_to_index = {str(row.video_id): idx for idx, row in test_manifest.reset_index(drop=True).iterrows()}
print(selected_video_ids)
display(selected_df)


## Grad-CAM Smoke Check

This checks one selected video and all requested layers before the full visualization loop. Grad-CAM is computed from the predicted normalized EF scalar, not from EF loss or ground truth.


In [ ]:
smoke_video_id = selected_video_ids[0]
smoke_idx = video_id_to_index[smoke_video_id]
smoke_batch = next(iter(DataLoader(Subset(test_dataset, [smoke_idx]), batch_size=1, shuffle=False, num_workers=0)))
smoke_video = smoke_batch["video"].to(device)
normal_pred = float(predictions_df[predictions_df["video_id"] == smoke_video_id].iloc[0]["ef_pred"])

for layer_name in config["gradcam_layers"]:
    result = r2plus1d_ef_gradcam(model, smoke_video, layer_name=layer_name, ef_mean=ef_mean, ef_std=ef_std)
    diff = abs(result.pred_ef - normal_pred)
    expected_native_t = int(config["expected_native_t_by_layer"][layer_name])
    overlay_alpha = float(config["overlay_alpha_by_layer"].get(layer_name, config["overlay_alpha"]))
    contour_level = config["motion_trace_contour_level_by_layer"].get(layer_name)
    print(f"[{layer_name}] normal EF={normal_pred:.6f} Grad-CAM EF={result.pred_ef:.6f} diff={diff:.8f}")
    print("  activation shape:", result.activation_shape, "gradient shape:", result.gradient_shape)
    print("  native CAM:", result.signed_native_cams.shape, "aligned CAM:", result.signed_raw_cams.shape)
    print("  max abs signed:", float(np.abs(result.signed_raw_cams).max()), "positive frame-norm max:", float(result.frame_normalized_positive_cams.max()))
    print("  zero-positive frames:", int((result.positive_cams.reshape(result.positive_cams.shape[0], -1).max(axis=1) <= 1e-12).sum()))
    print("  temporal interpolation applied:", result.temporal_interpolation_applied)
    print("  overlay alpha:", overlay_alpha)
    print("  thresholding applied:", contour_level is not None)
    assert diff < 1e-3
    assert result.signed_native_cams.shape[0] == expected_native_t, result.signed_native_cams.shape
    assert result.native_t == expected_native_t, result.native_t
    assert result.aligned_t == int(config["clip_length"]), result.aligned_t
    assert result.signed_raw_cams.shape == (int(config["clip_length"]), *tuple(config["image_size"]))
    assert result.positive_cams.shape == result.signed_raw_cams.shape
    assert np.isfinite(result.signed_raw_cams).all()
    assert np.isfinite(result.frame_normalized_positive_cams).all()
    assert float(np.abs(result.signed_raw_cams).max()) > 1e-12
    # Positive/ReLU CAM can be all-zero for a regression layer if the signed evidence is negative.
print("R(2+1)D Grad-CAM smoke check passed.")


## Generate Grad-CAM Outputs

For each selected video and each layer, this saves native CAM volumes, 32-frame visualization-aligned CAMs, diagnostics CSVs, static overlays, motion traces, and MP4 overlays for positive clip-normalized and signed robust-display variants only.

Layer 1 is the primary frame-level temporal-analysis layer because its native temporal resolution is 32. Layer 2 and layer 4 are useful localization views, but their 32-frame figures/videos are temporally interpolated visualizations, not 32 independent native explanations.


In [ ]:
overlay_specs = [
    {
        "suffix": "positive_frame_normalized_primary_aligned32",
        "cam_key": "frame_normalized_positive_cams",
        "overlay_name": "positive frame-normalized (primary temporal viewing; aligned32)",
        "signed": False,
        "cmap_name": "turbo",
    },
    {
        "suffix": "positive_clip_normalized_aligned32",
        "cam_key": "clip_normalized_cams",
        "overlay_name": "positive clip-normalized (cross-frame magnitude comparison only; aligned32)",
        "signed": False,
        "cmap_name": "turbo",
    },
    {
        "suffix": "signed_clip_normalized_aligned32",
        "cam_key": "signed_clip_normalized_cams",
        "overlay_name": "signed clip-normalized (aligned32)",
        "signed": True,
        "signed_display_mode": "faithful",
        "signed_display_percentile": None,
        "cmap_name": "coolwarm",
    },
    {
        "suffix": "signed_faithful_aligned32",
        "cam_key": "signed_raw_cams",
        "overlay_name": "signed raw faithful (aligned32)",
        "signed": True,
        "signed_display_mode": "faithful",
        "signed_display_percentile": None,
        "cmap_name": "coolwarm",
    },
    {
        "suffix": "signed_robust_display_aligned32",
        "cam_key": "signed_clip_normalized_cams",
        "overlay_name": "signed robust-display visualization only (aligned32)",
        "signed": True,
        "signed_display_mode": "enhanced",
        "signed_display_percentile": 99.0,
        "cmap_name": "coolwarm",
    },
]

# Layer 1 is the faithful frame-level temporal-analysis layer. Layer 2 and layer 4
# overlays are aligned to 32 frames for visualization, but their native temporal
# resolutions are lower, so their aligned centroids are visualization-space summaries.
manifest_rows = []
all_diagnostics = []
all_centroids = []
all_native_metrics = []
all_aligned_metrics = []

for video_id in tqdm(selected_video_ids, desc="R(2+1)D Grad-CAM videos"):
    idx = video_id_to_index[video_id]
    batch = next(iter(DataLoader(Subset(test_dataset, [idx]), batch_size=1, shuffle=False, num_workers=0)))
    video = batch["video"].to(device)
    pred_row = predictions_df[predictions_df["video_id"] == video_id].iloc[0]
    assert tuple(video.shape) == (1, 3, int(config["clip_length"]), *tuple(config["image_size"])), tuple(video.shape)

    for layer_name in config["gradcam_layers"]:
        layer_dir = RUN_DIR / layer_name
        expected_native_t = int(config["expected_native_t_by_layer"][layer_name])
        layer_alpha = float(config["overlay_alpha_by_layer"].get(layer_name, config["overlay_alpha"]))
        contour_level = config["motion_trace_contour_level_by_layer"].get(layer_name)
        thresholding_applied = contour_level is not None

        result = r2plus1d_ef_gradcam(model, video, layer_name=layer_name, ef_mean=ef_mean, ef_std=ef_std)
        assert abs(result.pred_ef - float(pred_row["ef_pred"])) < 1e-3, (result.pred_ef, float(pred_row["ef_pred"]))
        assert result.native_t == expected_native_t, (layer_name, result.native_t)
        assert result.aligned_t == int(config["clip_length"]), (layer_name, result.aligned_t)
        assert result.signed_native_cams.shape[0] == expected_native_t
        assert result.signed_raw_cams.shape == (int(config["clip_length"]), *tuple(config["image_size"]))
        assert result.positive_cams.shape == result.signed_raw_cams.shape
        assert np.isfinite(result.signed_native_cams).all()
        assert np.isfinite(result.signed_raw_cams).all()
        assert np.isfinite(result.frame_normalized_positive_cams).all()
        assert float(np.abs(result.signed_raw_cams).max()) > 1e-12
        # Positive/ReLU CAM can be all-zero for a regression layer if the signed evidence is negative.

        signed_native_before = result.signed_native_cams.copy()
        signed_aligned_before = result.signed_raw_cams.copy()
        npz_path = layer_dir / "npz" / f"{video_id}_{layer_name}_native_and_aligned32_gradcam.npz"
        diagnostics_csv = layer_dir / "diagnostics" / f"{video_id}_{layer_name}_aligned32_diagnostics.csv"
        row = save_r2plus1d_gradcam_npz(
            npz_path,
            result,
            batch,
            checkpoint_metadata=checkpoint_metadata,
            dataset_metrics=dataset_metrics,
            diagnostics_csv_path=diagnostics_csv,
        )

        native_metric_df = r2plus1d_cam_metric_table(
            result.signed_native_cams,
            result.positive_native_cams,
            layer_name=layer_name,
            space="native",
        )
        aligned_metric_df = r2plus1d_cam_metric_table(
            result.signed_raw_cams,
            result.positive_cams,
            layer_name=layer_name,
            space="aligned32_visualization",
        )
        for metric_df in [native_metric_df, aligned_metric_df]:
            metric_df.insert(0, "video_id", video_id)
            metric_df["native_t"] = result.native_t
            metric_df["aligned_t"] = result.aligned_t
            metric_df["temporal_interpolation_applied"] = result.temporal_interpolation_applied
            metric_df["display_only_enhancement_used_for_metrics"] = False
        assert not native_metric_df["display_only_enhancement_used_for_metrics"].any()
        assert not aligned_metric_df["display_only_enhancement_used_for_metrics"].any()
        native_metric_csv = layer_dir / "temporal_metrics_native" / f"{video_id}_{layer_name}_native_temporal_metrics.csv"
        aligned_metric_csv = layer_dir / "temporal_metrics_aligned32" / f"{video_id}_{layer_name}_aligned32_visualization_metrics.csv"
        native_metric_csv.parent.mkdir(parents=True, exist_ok=True)
        aligned_metric_csv.parent.mkdir(parents=True, exist_ok=True)
        native_metric_df.to_csv(native_metric_csv, index=False)
        aligned_metric_df.to_csv(aligned_metric_csv, index=False)
        all_native_metrics.append(native_metric_df)
        all_aligned_metrics.append(aligned_metric_df)

        overlay_paths = []
        for spec in overlay_specs:
            overlay_path = layer_dir / "overlays" / spec["suffix"] / f"{video_id}_{layer_name}_{spec['suffix']}_overlay.png"
            make_r2plus1d_overlay_figure(
                npz_path,
                overlay_path,
                dataset_metrics,
                cam_key=spec["cam_key"],
                overlay_name=spec["overlay_name"],
                signed=bool(spec["signed"]),
                signed_display_mode=spec.get("signed_display_mode", "faithful"),
                signed_display_percentile=spec.get("signed_display_percentile"),
                cmap_name=spec.get("cmap_name"),
                alpha=layer_alpha,
            )
            overlay_paths.append(str(overlay_path))

        heatmap_only_path = ""
        if layer_name == "layer1":
            heatmap_only_path = layer_dir / "heatmap_only" / "positive_frame_normalized_primary_aligned32" / f"{video_id}_{layer_name}_aligned32_positive_frame_normalized_heatmap_only.png"
            make_r2plus1d_heatmap_only_figure(
                npz_path,
                heatmap_only_path,
                dataset_metrics,
                cam_key="frame_normalized_positive_cams",
                overlay_name="positive frame-normalized heatmap-only diagnostic (aligned32)",
                cmap_name="turbo",
            )
            overlay_paths.append(str(heatmap_only_path))

        motion_trace_path = layer_dir / "motion_trace_overlays" / "frame_normalized_primary_aligned32" / f"{video_id}_{layer_name}_aligned32_motion_trace_overlay.png"
        centroid_df = make_r2plus1d_motion_trace_overlay(
            npz_path,
            motion_trace_path,
            dataset_metrics,
            cam_key="frame_normalized_positive_cams",
            overlay_name="frame-normalized motion trace (aligned32; primary only for layer1)",
            alpha=layer_alpha,
            contour_level=contour_level,
        )
        centroid_df.insert(0, "video_id", video_id)
        centroid_df.insert(1, "layer_name", layer_name)
        centroid_df["centroid_source"] = "frame_normalized_positive_cams"
        centroid_df["centroid_metric_space"] = "native_frame_level" if layer_name == "layer1" else "aligned32_visualization_interpolated"
        centroid_df["native_frame_level_temporal_analysis_valid"] = bool(layer_name == "layer1")
        centroid_df["native_t"] = result.native_t
        centroid_df["aligned_t"] = result.aligned_t
        centroid_csv = layer_dir / "centroid_diagnostics" / f"{video_id}_{layer_name}_aligned32_centroids.csv"
        centroid_csv.parent.mkdir(parents=True, exist_ok=True)
        centroid_df.to_csv(centroid_csv, index=False)
        overlay_paths.append(str(motion_trace_path))

        positive_video_path = layer_dir / "videos" / "positive_clip_normalized_aligned32" / f"{video_id}_{layer_name}_aligned32_positive_clip_normalized.mp4"
        make_r2plus1d_overlay_video(
            npz_path,
            positive_video_path,
            cam_key="clip_normalized_cams",
            signed=False,
            fps=int(config["video_fps"]),
            alpha=layer_alpha,
        )
        signed_video_path = layer_dir / "videos" / "signed_robust_display_aligned32" / f"{video_id}_{layer_name}_aligned32_signed_robust_display.mp4"
        make_r2plus1d_overlay_video(
            npz_path,
            signed_video_path,
            cam_key="signed_clip_normalized_cams",
            signed=True,
            signed_display_percentile=99.0,
            fps=int(config["video_fps"]),
            alpha=layer_alpha,
        )

        assert np.array_equal(signed_native_before, result.signed_native_cams)
        assert np.array_equal(signed_aligned_before, result.signed_raw_cams)
        with np.load(npz_path, allow_pickle=False) as saved:
            assert np.array_equal(saved["signed_native_cams"], signed_native_before)
            assert np.array_equal(saved["signed_raw_cams"], signed_aligned_before)
            assert int(saved["native_t"]) == expected_native_t
            assert int(saved["aligned_t"]) == int(config["clip_length"])

        row["overlay_paths"] = json.dumps(overlay_paths)
        row["primary_overlay_path"] = overlay_paths[0]
        row["heatmap_only_path"] = str(heatmap_only_path)
        row["motion_trace_overlay_path"] = str(motion_trace_path)
        row["positive_clip_normalized_video_path"] = str(positive_video_path)
        row["signed_robust_video_path"] = str(signed_video_path)
        row["centroid_csv_path"] = str(centroid_csv)
        row["native_metric_csv_path"] = str(native_metric_csv)
        row["aligned32_visualization_metric_csv_path"] = str(aligned_metric_csv)
        row["overlay_alpha_used"] = layer_alpha
        row["thresholding_applied"] = thresholding_applied
        row["motion_trace_contour_level"] = contour_level if contour_level is not None else "disabled"
        row["primary_frame_level_temporal_analysis_layer"] = bool(layer_name == "layer1")
        manifest_rows.append(row)

        diagnostics_df = pd.read_csv(diagnostics_csv)
        diagnostics_df["npz_path"] = str(npz_path)
        diagnostics_df["native_t"] = result.native_t
        diagnostics_df["aligned_t"] = result.aligned_t
        diagnostics_df["temporal_interpolation_applied"] = result.temporal_interpolation_applied
        all_diagnostics.append(diagnostics_df)
        all_centroids.append(centroid_df)

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(MANIFEST_DIR / "r2plus1d_gradcam_manifest.csv", index=False)
all_diagnostics_df = pd.concat(all_diagnostics, ignore_index=True) if all_diagnostics else pd.DataFrame()
all_diagnostics_df.to_csv(MANIFEST_DIR / "r2plus1d_gradcam_temporal_diagnostics.csv", index=False)
all_centroids_df = pd.concat(all_centroids, ignore_index=True) if all_centroids else pd.DataFrame()
all_centroids_df.to_csv(MANIFEST_DIR / "r2plus1d_gradcam_centroid_diagnostics.csv", index=False)
all_native_metrics_df = pd.concat(all_native_metrics, ignore_index=True) if all_native_metrics else pd.DataFrame()
all_native_metrics_df.to_csv(MANIFEST_DIR / "r2plus1d_gradcam_native_temporal_metrics.csv", index=False)
all_aligned_metrics_df = pd.concat(all_aligned_metrics, ignore_index=True) if all_aligned_metrics else pd.DataFrame()
all_aligned_metrics_df.to_csv(MANIFEST_DIR / "r2plus1d_gradcam_aligned32_visualization_metrics.csv", index=False)
display(manifest_df.head())


## Required Outputs


In [ ]:
expected_outputs = [
    RUN_DIR / "config.json",
    RUN_DIR / "test_metrics.json",
    MANIFEST_DIR / "test_manifest.csv",
    MANIFEST_DIR / "normal_test_predictions.csv",
    MANIFEST_DIR / "selected_gradcam_videos.csv",
    MANIFEST_DIR / "r2plus1d_gradcam_manifest.csv",
    MANIFEST_DIR / "r2plus1d_gradcam_temporal_diagnostics.csv",
    MANIFEST_DIR / "r2plus1d_gradcam_centroid_diagnostics.csv",
    MANIFEST_DIR / "r2plus1d_gradcam_native_temporal_metrics.csv",
    MANIFEST_DIR / "r2plus1d_gradcam_aligned32_visualization_metrics.csv",
]
for path in expected_outputs:
    print(path, "exists=", path.exists())

summary = {
    "run_mode": RUN_MODE,
    "selected_video_count": len(selected_video_ids),
    "layers": list(config["gradcam_layers"]),
    "layer_resolution_labels": [f"{layer} | native T={config['expected_native_t_by_layer'][layer]} | aligned T={config['clip_length']}" for layer in config["gradcam_layers"]],
    "overlay_alpha_by_layer": config["overlay_alpha_by_layer"],
    "output_dir": str(RUN_DIR),
    "test_metrics": dataset_metrics,
    "temporal_interpolation_note": "Native CAM volumes are saved separately. The 32-frame signed_raw_cams/positive_cams are aligned to input frames for visualization. Layer 2 and layer 4 aligned32 maps are temporally interpolated and are not 32 independent native explanations.",
    "primary_temporal_analysis_layer": "layer1 only, because its native temporal resolution is 32.",
    "mp4_outputs": "Generated only for positive clip-normalized and signed robust-display variants.",
}
with (MANIFEST_DIR / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
